In [1]:
!pip install qwen_vl_utils
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",device_map="cuda")
from PIL import Image
import torch
# default: Load the model on the available device(s)
# model = Qwen2VLForConditionalGeneration.from_pretrained(
#     # "Qwen/Qwen2-VL-32B-Instruct",
#     "Qwen/Qwen2.5-VL-3B-Instruct",
#     # device_map="cuda",
# )

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen2VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2-VL-7B-Instruct",
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384. You can set min_pixels and max_pixels according to your needs, such as a token count range of 256-1280, to balance speed and memory usage.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
import os
folder_path = '/content/frames'

# List all files in the folder
files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

print(files)

['frame_00300_t6.00.jpg', 'frame_05200_t104.00.jpg', 'frame_02100_t42.00.jpg', 'frame_01200_t24.00.jpg', 'frame_02900_t58.00.jpg', 'frame_03800_t76.00.jpg', 'frame_01000_t20.00.jpg', 'frame_01250_t25.00.jpg', 'frame_03200_t64.00.jpg', 'frame_00850_t17.00.jpg', 'frame_02200_t44.00.jpg', 'frame_01900_t38.00.jpg', 'frame_02800_t56.00.jpg', 'frame_00550_t11.00.jpg', 'frame_05450_t109.00.jpg', 'frame_05700_t114.00.jpg', 'frame_03700_t74.00.jpg', 'frame_02750_t55.00.jpg', 'frame_02650_t53.00.jpg', 'frame_01100_t22.00.jpg', 'frame_01150_t23.00.jpg', 'frame_02350_t47.00.jpg', 'frame_05900_t118.00.jpg', 'frame_04700_t94.00.jpg', 'frame_05100_t102.00.jpg', 'frame_05300_t106.00.jpg', 'frame_00450_t9.00.jpg', 'frame_00700_t14.00.jpg', 'frame_02400_t48.00.jpg', 'frame_04500_t90.00.jpg', 'frame_01300_t26.00.jpg', 'frame_05250_t105.00.jpg', 'frame_04250_t85.00.jpg', 'frame_02700_t54.00.jpg', 'frame_00100_t2.00.jpg', 'frame_01550_t31.00.jpg', 'frame_04950_t99.00.jpg', 'frame_01600_t32.00.jpg', 'frame_

In [4]:
for im in files:
  image_path =  '/content/frames/' + im
  image = Image.open(image_path).convert("RGB") # Ensure the image is in RGB format

  messages = [
      {
          "role": "user",
          "content": [
              {
                  "type": "image",
                  "image": image,
              },
              {"type": "text", "text": "i need your answer to be in this form {the_action_in_the_frame : action type,players : [{team : team color,player_num : num},{team :  team color,player_num : num}] }"},
          ],
      }
  ]

  # Preparation for inference
  text = processor.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )
  image_inputs, video_inputs = process_vision_info(messages)
  inputs = processor(
      text=[text],
      images=image_inputs,
      videos=video_inputs,
      padding=True,
      return_tensors="pt",
  )
  inputs = inputs.to("cuda")

  # Inference: Generation of the output
  generated_ids = model.generate(**inputs, max_new_tokens=128)
  generated_ids_trimmed = [
      out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
  ]
  output_text = processor.batch_decode(
      generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
  )
  print(output_text)


OutOfMemoryError: CUDA out of memory. Tried to allocate 442.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 18.12 MiB is free. Process 80768 has 14.72 GiB memory in use. Of the allocated memory 14.54 GiB is allocated by PyTorch, and 54.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [14]:
!pip install pytube
!pip install opencv-python
!pip install yt-dlp

In [15]:
import yt_dlp

def download_youtube_video(url, save_path="video.mp4"):
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]',
        'outtmpl': save_path,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    return save_path

# Example usage
video_path = download_youtube_video("https://youtu.be/pBNkcCWbjTk?si=Js1ObZcYiynSR_Ai")
print(f"Downloaded: {video_path}")


[youtube] Extracting URL: https://youtu.be/pBNkcCWbjTk?si=Js1ObZcYiynSR_Ai
[youtube] pBNkcCWbjTk: Downloading webpage
[youtube] pBNkcCWbjTk: Downloading tv client config
[youtube] pBNkcCWbjTk: Downloading player 73381ccc-main
[youtube] pBNkcCWbjTk: Downloading tv player API JSON
[youtube] pBNkcCWbjTk: Downloading ios player API JSON
[youtube] pBNkcCWbjTk: Downloading m3u8 information
[info] pBNkcCWbjTk: Downloading 1 format(s): 399+140
[download] video.mp4 has already been downloaded
Downloaded: video.mp4


In [16]:
import cv2

video_path = "video.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Cannot open video file.")
else:
    print("Video successfully opened.")

cap.release()


Video successfully opened.


In [17]:
import cv2

video_path = "video.mp4"
cap = cv2.VideoCapture(video_path)

if cap.isOpened():
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"FPS: {fps}, Total Frames: {total_frames}")
else:
    print("Error: Cannot open video file.")

cap.release()

FPS: 50.0, Total Frames: 6000


In [18]:
!apt-get install -y ffmpeg  # Ensure ffmpeg is installed
!ffmpeg -i video.mp4 -vcodec libx264 -preset ultrafast converted.mp4

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq -

In [19]:
import cv2

def extract_frames(video_path, frame_interval=1):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Cannot open video file.")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video FPS: {fps}, Total Frames: {total_frames}")  # Debugging info

    frame_count = 0
    frames = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("End of video reached.")
            break

        if frame_count % int(fps * frame_interval) == 0:
            timestamp = frame_count / fps
            frames.append((frame, timestamp))
            print(f"Extracted frame at {timestamp:.2f}s")  # Debugging output

        frame_count += 1

    cap.release()
    print(f"Total frames extracted: {len(frames)}")
    return frames

frames = extract_frames("converted.mp4")


Error: Cannot open video file.


In [20]:
import os
import cv2
import yt_dlp
import numpy as np

# ------------------------ Step 1: Download YouTube Video ------------------------
def download_youtube_video(url, output_filename="video.mp4"):
    ydl_opts = {
        'format': 'bestvideo+bestaudio/best',
        'outtmpl': output_filename,
        'quiet': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return output_filename

# ------------------------ Step 2: Convert Video for Compatibility ------------------------
def convert_video(input_path, output_path="converted.mp4"):
    os.system(f"ffmpeg -i {input_path} -vcodec libx264 -preset ultrafast {output_path}")
    return output_path

# ------------------------ Step 3: Extract Frames ------------------------
def extract_frames(video_path, output_folder="frames", frame_interval=50):
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"🎥 Video FPS: {fps}, Total Frames: {total_frames}")

    frame_count = 0
    extracted_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break  # End of video

        if frame_count % frame_interval == 0:
            timestamp = frame_count / fps  # Time in seconds
            frame_filename = f"{output_folder}/frame_{frame_count:05d}_t{timestamp:.2f}.jpg"
            cv2.imwrite(frame_filename, frame)
            extracted_count += 1

        frame_count += 1

    cap.release()
    print(f"✅ Total frames extracted: {extracted_count}")

# ------------------------ Step 4: Full Pipeline Execution ------------------------
def process_video(youtube_url):
    print("📥 Downloading video...")
    raw_video = download_youtube_video(youtube_url)

    print("🎬 Converting video format...")
    converted_video = convert_video(raw_video)

    print("🖼️ Extracting frames...")
    extract_frames(converted_video, frame_interval=50)  # Adjust interval as needed

    print("🚀 Processing complete!")


In [21]:

# ------------------------ Run the Pipeline ------------------------
youtube_url = "https://youtu.be/pBNkcCWbjTk?si=Js1ObZcYiynSR_Ai"  # Replace with actual video URL
process_video(youtube_url)


📥 Downloading video...
🎬 Converting video format...
🖼️ Extracting frames...
🎥 Video FPS: 0.0, Total Frames: 0
✅ Total frames extracted: 0
🚀 Processing complete!


In [22]:
!pip install transformers accelerate torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [23]:
from transformers import BlipProcessor, BlipForConditionalGeneration, BertTokenizerFast
import torch
from PIL import Image
import os
import json

# Load BLIP-2 Model
processor = BlipProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b").to("cuda" if torch.cuda.is_available() else "cpu")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'BertTokenizerFast'.


TypeError: the JSON object must be str, bytes or bytearray, not NoneType

In [ ]:
def analyze_frames(frame_folder="frames", output_file="descriptions.txt"):
    frames = sorted(os.listdir(frame_folder))  # Sort frames in order
    results = []

    for frame in frames:
        frame_path = os.path.join(frame_folder, frame)
        image = Image.open(frame_path).convert("RGB")

        # Prepare input for BLIP-2
        inputs = processor(images=image, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

        # Generate caption
        with torch.no_grad():
            caption = model.generate(**inputs)
            description = processor.decode(caption[0], skip_special_tokens=True)

        # Extract timestamp from filename
        timestamp = frame.split("_t")[1].replace(".jpg", "")
        results.append(f"[{timestamp}s] {description}")
        print(f"📝 {timestamp}s → {description}")

    # Save results
    with open(output_file, "w") as f:
        f.write("\n".join(results))

    print(f"✅ Saved descriptions to {output_file}")

# Run the frame analysis
analyze_frames()